In [1]:
import inspect
import json

import cmasher as cmr
import gc_utils
import h5py
import halo_analysis as halo
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy.interpolate import griddata, interp1d
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.stats import gaussian_kde, lognorm
from sklearn.svm import SVC, LinearSVC

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [2]:
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]


sim_dir = "/Users/z5114326/Documents/simulations/"
# sim_dir = "/Volumes/Expansion/simulations/"

In [3]:
sim = "m12c"

ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
ghost_data = h5py.File(ghost_file, "r")

for it_id in ghost_data.keys():
    msk1 = ghost_data[it_id]["source"]["grpid"][()] == 0
    # msk2 = ghost_data[it_id]["source"]["s_flag"][()].astype(bool)
    msk2 = np.ones(len(msk1), dtype=bool)
    print(np.unique(ghost_data[it_id]["source"]["tacc"][msk1 & msk2], return_counts=True))

(array([-1.   ,  0.378]), array([5768,    1]))
(array([-1.]), array([7082]))
(array([-1.   ,  0.378]), array([3192,    1]))
(array([-1.   ,  0.378]), array([6953,    2]))
(array([-1.   ,  0.378]), array([6795,    1]))
(array([-1.   ,  0.378]), array([6462,    1]))
(array([-1.   ,  0.378]), array([11748,     1]))
(array([-1.   ,  0.378]), array([6104,    2]))
(array([-1.   ,  0.378]), array([3995,    2]))
(array([-1.   ,  0.378]), array([7366,    2]))
(array([-1.   ,  0.378]), array([7851,    1]))
(array([-1.   ,  0.378]), array([8025,    2]))
(array([-1.   ,  0.378]), array([7136,    2]))
(array([-1.   ,  0.378]), array([4419,    1]))
(array([-1.   ,  0.378]), array([11409,     2]))
(array([-1.   ,  0.378]), array([11741,     2]))
(array([-1.   ,  0.378]), array([6931,    1]))
(array([-1.]), array([4603]))
(array([-1.   ,  0.378]), array([13157,     1]))
(array([-1.   ,  0.378]), array([3866,    1]))
(array([-1.   ,  0.378]), array([9383,    2]))
(array([-1.   ,  0.378]), array([5006, 

In [4]:
sim = "m12m"

proc_file = f"{sim_dir}{sim}/{sim}_processed.hdf5"
proc_data = h5py.File(proc_file, "r")

for it_id in proc_data.keys():
    msk1 = proc_data[it_id]["source"]["group_id"][()] == 0
    # msk2 = ghost_data[it_id]["source"]["survive_flag"][()].astype(bool)
    msk2 = np.ones(len(msk1), dtype=bool)
    print(np.unique(proc_data[it_id]["source"]["t_acc"][msk1 & msk2], return_counts=True))

(array([-1.   ,  0.378]), array([12742,     1]))
(array([-1.   ,  0.378]), array([8535,    1]))
(array([-1.   ,  0.378]), array([16495,     4]))
(array([-1.   ,  0.378]), array([12248,     3]))
(array([-1.   ,  0.378]), array([9749,    3]))
(array([-1.   ,  0.378]), array([9370,    2]))
(array([-1.   ,  0.378]), array([8481,    1]))
(array([-1.   ,  0.378]), array([7392,    3]))
(array([-1.   ,  0.378]), array([10733,     2]))
(array([-1.]), array([7068]))
(array([-1.   ,  0.378]), array([11668,     2]))
(array([-1.   ,  0.378]), array([10119,     3]))
(array([-1.   ,  0.378]), array([11795,     2]))
(array([-1.]), array([10403]))
(array([-1.   ,  0.378]), array([9895,    2]))
(array([-1.   ,  0.378]), array([5858,    2]))
(array([-1.   ,  0.378]), array([10962,     1]))
(array([-1.   ,  0.378]), array([7513,    2]))
(array([-1.   ,  0.378]), array([7405,    2]))
(array([-1.   ,  0.378]), array([8745,    3]))
(array([-1.   ,  0.378]), array([16913,     3]))
(array([-1.   ,  0.378]), ar

In [5]:
# All snpas index (snapshot) 11 is 0.37816179
# For some reason defaulats to that

# Probably from a bug in process_data.py:
# The following lines exist

# get the time of accretion
# snap_acc = halt["snapshot"][desc_lst[idx_acc]]
# t_acc = snap_all["time[Gyr]"][snap_acc]
# t_acc = np.round(t_acc, 3)

# max ever count as tacc != -1 is 4 across all sims and all iterations
# none ever survive to z=0